# Pre-processing News Data


## Bài toán

Dữ liệu gồm n văn bản phân vào 10 chủ đề khác nhau. Cần biễu diễn mỗi văn bản dưới dạng một vector số thể hiện cho nội dụng của văn bản đó.


## Sử dụng phương pháp mã hóa: TF-IDF

Cho tập gồm $n$ văn bản: $D = \{d_1, d_2, ... d_n\}$. Tập từ điển tương ứng được xây dựng từ $n$ văn bản này có độ dài là $m$

- Xét văn bản $d$ có $|d|$ từ và $t$ là một từ trong $d$. Mã hóa tf-idf của $t$ trong văn bản $d$ được biểu diễn:
  \begin{equation}
  \begin{split}
  \text{tf}_{t, d} &= \frac{f_t}{|d|} \\
  \text{idf}_{t, d} &= \log\frac{n}{n*t}, \ \ \ \ n_t = |\{d\in D: t\in d\}| \\
  \text{tf-idf}*{t d} &= \text{tf}_{t, d} \times \text{idf}_{t, d}
  \end{split}
  \end{equation}

- Khi đó văn bản $d$ được mã hóa là một vector $m$ chiều. Các từ xuất hiện trong d sẽ được thay bằng giá trị tf-idf tương ứng. Các từ không xuất hiện trong $d$ thì thay là 0


# Các bước làm


## Chuẩn bị các thư viện cần thiết


In [1]:
import os
import matplotlib.pyplot as plt
import numpy as np

from sklearn.datasets import load_files
from pyvi import ViTokenizer
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfTransformer
from sklearn.pipeline import Pipeline

%matplotlib inline

## Load dữ liệu từ thư mục

Cấu trúc thư mục như sau

- data/news_vnexpress/
  - Kinh tế:
    - bài báo 1.txt
    - bài báo 2.txt
  - Pháp luật
    - bài báo 3.txt
    - bài báo 4.txt


In [2]:
# Có thể chỉnh lại đường dẫn thư mục cho phù hợp
INPUT = "news_vnexpress"
os.makedirs(
    "images", exist_ok=True
)  # thư mục lưu các các hình ảnh trong quá trình huấn luyện và đánh gía

In [3]:
# statistics
print("Các nhãn và số văn bản tương ứng trong dữ liệu")
print("----------------------------------------------")
n = 0
for label in os.listdir(INPUT):
    print(f"{label}: {len(os.listdir(os.path.join(INPUT, label)))}")
    n += len(os.listdir(os.path.join(INPUT, label)))

print("-------------------------")
print(f"Tổng số văn bản: {n}")

Các nhãn và số văn bản tương ứng trong dữ liệu
----------------------------------------------
giai-tri: 201
kinh-doanh: 262
suc-khoe: 162
phap-luat: 59
du-lich: 54
the-thao: 173
doi-song: 120
khoa-hoc: 144
thoi-su: 59
giao-duc: 105
-------------------------
Tổng số văn bản: 1339


In [4]:
# load data
data_train = load_files(container_path=INPUT, encoding="utf-8")
print("mapping:")
for i in range(len(data_train.target_names)):
    print(f"{data_train.target_names[i]} - {i}")

print("--------------------------")
print(data_train.filenames[0:1])
# print(data_train.data[0:1])
print(data_train.target[0:1])
print(data_train.data[0:1])

print("\nTổng số  văn bản: {}".format(len(data_train.filenames)))

mapping:
doi-song - 0
du-lich - 1
giai-tri - 2
giao-duc - 3
khoa-hoc - 4
kinh-doanh - 5
phap-luat - 6
suc-khoe - 7
the-thao - 8
thoi-su - 9
--------------------------
['news_vnexpress/khoa-hoc/00133.txt']
[4]
['Mời độc giả đặt câu hỏi tại đây\n']

Tổng số  văn bản: 1339


## Chuyển dữ liệu dạng text về ma trận (n x m) bằng TF-IDF


- Bạn cần viết đoạn mã tương ứng trong cell bên dưới. Theo các bước được gợi ý


In [5]:
# load dữ liệu các stopwords
# ---> Code ở đây
with open("vietnamese-stopwords.txt", "r", encoding="utf-8") as f:
    stopwords = [line.strip().replace(" ", "_") for line in f if line.strip()]

# Chuyển hoá dữ liệu text về dạng vector TF
#     - loại bỏ từ dừng
#     - sinh từ điển
# ---> Code ở đây

X_tokenized = [ViTokenizer.tokenize(doc) for doc in data_train.data]

module_count_vector = CountVectorizer(stop_words=stopwords)
X_counts = module_count_vector.fit_transform(X_tokenized)

# Hàm thực hiện chuyển đổi dữ liệu text thành dữ liệu số dạng ma trận
# Input: Dữ liệu 2 chiều dạng numpy.array, mảng nhãn id dạng numpy.array
# data_preprocessed = #Code ở đây

tfidf_transformer = TfidfTransformer()
data_preprocessed = tfidf_transformer.fit_transform(X_counts)

X = data_preprocessed  # thuoc tinh
Y = data_train.target  # nhan

print(f"\nSố lượng từ trong từ điển: {len(module_count_vector.vocabulary_)}")
print(f"Kích thước dữ liệu sau khi xử lý: {X.shape}")
print(f"Kích thước nhãn tương ứng: {Y.shape}")


Số lượng từ trong từ điển: 24351
Kích thước dữ liệu sau khi xử lý: (1339, 24351)
Kích thước nhãn tương ứng: (1339,)


In [6]:
print(X[100].toarray())
print(Y[100])

[[0. 0. 0. ... 0. 0. 0.]]
5


In [7]:
print(X[100])  # Sau khi xử lí, dữ liệu được lưu dưới dạng ma trận thưa như sau:

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 217 stored elements and shape (1, 24351)>
  Coords	Values
  (0, 78)	0.015478703397323663
  (0, 97)	0.021134250221654934
  (0, 153)	0.023671358001332393
  (0, 185)	0.04805423414913507
  (0, 266)	0.03382405090314814
  (0, 386)	0.025219032877002084
  (0, 391)	0.03483406741576265
  (0, 412)	0.04977704283322189
  (0, 656)	0.023169757707443596
  (0, 902)	0.03376491200960838
  (0, 1221)	0.05206883654806034
  (0, 1247)	0.10413767309612068
  (0, 1260)	0.05206883654806034
  (0, 1745)	0.04722759863952526
  (0, 1792)	0.026931200395042987
  (0, 1970)	0.07618781961894648
  (0, 2114)	0.05206883654806034
  (0, 2393)	0.07445161124035883
  (0, 2586)	0.030026550146382367
  (0, 2755)	0.04602061487260788
  (0, 3040)	0.029743417719199822
  (0, 3178)	0.033058969499355634
  (0, 3186)	0.045713644403010366
  (0, 3330)	0.034550934988580105
  (0, 3641)	0.02631231276425594
  :	:
  (0, 22489)	0.025901791695506483
  (0, 22504)	0.02933927731449147
  (0, 22